In [1]:
import json
import time
from sentence_transformers import SentenceTransformer, util


INPUT_FILE = "dataset_sample_10k.json"
OUTPUT_FILE = "semantic_relevance_results.json"

MODELS = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2"
]

# HELPER FUNCTIONS

def normalize_keywords(keywords):
    if isinstance(keywords, list):
        return [k.lower().strip() for k in keywords if isinstance(k, str)]
    elif isinstance(keywords, str):
        return [keywords.lower().strip()]
    return []


def get_title_text(item):
    return item.get("title", "")


def get_description_text(item):
    return item.get("description", "")


def run_relevance_evaluation(model_name, data, title_texts, description_texts, keyword_lists):
    print(f"\n{'=' * 70}")
    print(f"Loading model: {model_name}")
    print(f"{'=' * 70}")

    model = SentenceTransformer(model_name)

    # COMPUTING EMBEDDINGS 
    print("🔄 Computing title/description embeddings...")
    embed_start = time.time()

    title_embeddings = model.encode(
        title_texts,
        convert_to_tensor=True,
        batch_size=64,
        show_progress_bar=True
    )

    description_embeddings = model.encode(
        description_texts,
        convert_to_tensor=True,
        batch_size=64,
        show_progress_bar=True
    )

    embed_end = time.time()
    print(f"✅ Base embeddings computed in {embed_end - embed_start:.2f}s\n")

    #  SIMILARITY 
    print("🔄 Computing semantic relevance...\n")

    model_results = []

    for i in range(len(data)):
        title_emb = title_embeddings[i]
        description_emb = description_embeddings[i]
        keywords = keyword_lists[i]

        if len(keywords) > 0:
            keyword_embs = model.encode(
                keywords,
                convert_to_tensor=True,
                batch_size=32,
                show_progress_bar=False
            )

            # Aggregated similarity
            keyword_emb = keyword_embs.mean(dim=0)

            agg_title_sim = float(
                util.cos_sim(keyword_emb, title_emb)[0][0]
            )

            agg_description_sim = float(
                util.cos_sim(keyword_emb, description_emb)[0][0]
            )

            # Per-keyword similarity
            title_sims = util.cos_sim(
                keyword_embs,
                title_emb
            ).cpu().numpy()

            avg_title_sim = float(title_sims.mean())

            description_sims = util.cos_sim(
                keyword_embs,
                description_emb
            ).cpu().numpy()

            avg_description_sim = float(description_sims.mean())

        else:
            agg_title_sim = 0.0
            agg_description_sim = 0.0
            avg_title_sim = 0.0
            avg_description_sim = 0.0

        model_results.append({
            "dataset_id": data[i].get("dataset_id"),

            "agg_title_similarity": agg_title_sim,
            "agg_description_similarity": agg_description_sim,

            "avg_title_similarity": avg_title_sim,
            "avg_description_similarity": avg_description_sim,

            "num_keywords": len(keywords)
        })

        if (i + 1) % 500 == 0:
            print(f"[{model_name}] Processed {i+1}/{len(data)} datasets")

    return model_results

# MAIN
if __name__ == "__main__":
    start_time = time.time()

    print("🔄 Loading dataset...\n")

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"Loaded {len(data)} datasets\n")

    #  PREPARE TEXT
    title_texts = []
    description_texts = []
    keyword_lists = []

    for d in data:
        keywords = normalize_keywords(d.get("keywords", []))
        keyword_lists.append(keywords)

        title_texts.append(get_title_text(d))
        description_texts.append(get_description_text(d))

    all_results = {}

    for model_name in MODELS:
        model_results = run_relevance_evaluation(
            model_name,
            data,
            title_texts,
            description_texts,
            keyword_lists
        )

        all_results[model_name] = model_results

    # SAVE 
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2)

    print(f"\n💾 Results saved to {OUTPUT_FILE}")

    end_time = time.time()
    print(f"\n⏱ Total time: {end_time - start_time:.2f}s")

🔄 Loading dataset...

Loaded 10000 datasets


Loading model: sentence-transformers/all-MiniLM-L6-v2
🔄 Computing title/description embeddings...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

✅ Base embeddings computed in 616.61s

🔄 Computing semantic relevance...

[sentence-transformers/all-MiniLM-L6-v2] Processed 500/10000 datasets
[sentence-transformers/all-MiniLM-L6-v2] Processed 1000/10000 datasets
[sentence-transformers/all-MiniLM-L6-v2] Processed 1500/10000 datasets
[sentence-transformers/all-MiniLM-L6-v2] Processed 2000/10000 datasets
[sentence-transformers/all-MiniLM-L6-v2] Processed 2500/10000 datasets
[sentence-transformers/all-MiniLM-L6-v2] Processed 3000/10000 datasets
[sentence-transformers/all-MiniLM-L6-v2] Processed 3500/10000 datasets
[sentence-transformers/all-MiniLM-L6-v2] Processed 4000/10000 datasets
[sentence-transformers/all-MiniLM-L6-v2] Processed 4500/10000 datasets
[sentence-transformers/all-MiniLM-L6-v2] Processed 5000/10000 datasets
[sentence-transformers/all-MiniLM-L6-v2] Processed 5500/10000 datasets
[sentence-transformers/all-MiniLM-L6-v2] Processed 6000/10000 datasets
[sentence-transformers/all-MiniLM-L6-v2] Processed 6500/10000 datasets
[sen

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔄 Computing title/description embeddings...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

✅ Base embeddings computed in 2960.26s

🔄 Computing semantic relevance...

[sentence-transformers/all-mpnet-base-v2] Processed 500/10000 datasets
[sentence-transformers/all-mpnet-base-v2] Processed 1000/10000 datasets
[sentence-transformers/all-mpnet-base-v2] Processed 1500/10000 datasets
[sentence-transformers/all-mpnet-base-v2] Processed 2000/10000 datasets
[sentence-transformers/all-mpnet-base-v2] Processed 2500/10000 datasets
[sentence-transformers/all-mpnet-base-v2] Processed 3000/10000 datasets
[sentence-transformers/all-mpnet-base-v2] Processed 3500/10000 datasets
[sentence-transformers/all-mpnet-base-v2] Processed 4000/10000 datasets
[sentence-transformers/all-mpnet-base-v2] Processed 4500/10000 datasets
[sentence-transformers/all-mpnet-base-v2] Processed 5000/10000 datasets
[sentence-transformers/all-mpnet-base-v2] Processed 5500/10000 datasets
[sentence-transformers/all-mpnet-base-v2] Processed 6000/10000 datasets
[sentence-transformers/all-mpnet-base-v2] Processed 6500/10000